# SVD Quantization on GPU

Run SVD-based sub-1-bit quantization with GPU acceleration.

**Runtime**: Runtime > Change runtime type > **GPU** (T4 or better)

**Expected time**: ~30-60 minutes depending on model size

In [ ]:
# Check GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Clone/pull the repo
!git clone https://github.com/toxzak-svg/Quantization-Exploration.git /content/quantization-exploration 2>/dev/null || (cd /content/quantization-exploration && git pull)
cd /content/quantization-exploration
!git pull origin main

In [ ]:
# Install dependencies
!pip install torch transformers accelerate numpy scikit-learn -q

In [ ]:
# Download Gemma 4 E2B model
from huggingface_hub import snapshot_download
import os

os.environ["HF_TOKEN"] = "your_hf_token_here"  # Replace with your token

MODEL_DIR = "/content/models/gemma-4-E2B"
snapshot_download(
    repo_id="google/gemma-4-2b-it",
    local_dir=MODEL_DIR,
    ignore_patterns=["*.gguf", "*.bin"]
)

In [ ]:
# Run SVD quantization at 60% threshold
!python scripts/quantize_svd_proper_v2.py \
    --model-dir /content/models/gemma-4-E2B \
    --output quantized/gemma_svd_60.pt \
    --threshold 0.60

In [ ]:
# Also try 70% threshold for comparison
!python scripts/quantize_svd_proper_v2.py \
    --model-dir /content/models/gemma-4-E2B \
    --output quantized/gemma_svd_70.pt \
    --threshold 0.70

In [ ]:
# Evaluate reconstruction quality
!python scripts/eval_reconstruction.py \
    --model-dir /content/models/gemma-4-E2B \
    --max-layers 100

In [ ]:
# Copy results to Google Drive
import shutil
import os

os.makedirs("/content/drive/MyDrive/quantization-results", exist_ok=True)
for f in ["quantized/gemma_svd_60.pt", "quantized/gemma_svd_70.pt"]:
    if os.path.exists(f):
        shutil.copy(f, "/content/drive/MyDrive/quantization-results/")
        print(f"Copied {f}")